# An OpenAI Agents SDK agent, trading a market that never existed

**The OpenAI Agents SDK** is the agent framework. **Tradefloor** is the
simulated market that measures what the agent does.

The agent below is an ordinary `agents.Agent`: a name, instructions, a model.
Nothing about it is Tradefloor-specific, and the adapter does not rebuild it
-- it binds a decision contract onto a *clone* and sends the observation as a
message, so the object you would run anywhere else is the object being
measured here.

What this notebook shows, in order: what the model is actually sent, what it
decided and why, how the wording of its instructions changed how far it
missed a limit by, and what it was never allowed to see.

A backtest on real data cannot say any of that. The model has read the whole
internet, and it has not read this market, because this market was generated
rather than recorded.

## 1. Live calls, and the recording

Tradefloor's market is deterministic. A language model behind an API is not,
so two live runs give two different agents and neither is reproducible.

This notebook **replays a recording by default and calls the model only when
asked to**. Three conditions gate a live run, and the opt-in comes first:
`TRADEFLOOR_LIVE_EXAMPLES=1`, then the API key, then the SDK. A credential
sitting in the environment is not consent to spend it -- without that gate,
anyone with a key exported who ran the slow test suite would re-execute every
notebook live and pay for it, having asked for neither.

The market re-executes for real either way; only the agent's answers come
from the recording, keyed by a digest of the exact input it was sent.

The committed recording is a genuine run. Nothing in it was written by hand.

In [1]:
import json
import sys
from pathlib import Path

import tradefloor as tf
from tradefloor.integrations.common import Transcript

# The production adapter. This notebook implements none of the integration:
# `tradefloor/integrations/openai_agents.py` is the source of truth, and this
# is the same object the shipped example and the test suite use.
from tradefloor.integrations.openai_agents import (BRIEF, OpenAIAgentsAdapter,
                                                   payload_of)

# Constants come from the example script beside this notebook, for the same
# reason: one definition, so the two cannot drift into describing different
# experiments.
sys.path.insert(0, str(Path.cwd()))
import five_days as example

LIVE = example.can_run_live()

print(f"tradefloor {tf.__version__}")
print(f"mode        {'live, calling the model' if LIVE else 'replay'}")
if LIVE:
    # The ceiling, printed BEFORE the run cell spends anything. One turn is
    # one model call, and the budget is a constant so this estimate and the
    # call it describes cannot drift apart.
    print(f"            {example.LIVE_MODEL}, {example.DAYS} decisions, "
          f"up to {example.LIVE_MAX_TURNS} model calls each")
    print(f"            ceiling {example.DAYS * example.LIVE_MAX_TURNS} "
          f"calls; this spends real money")
else:
    print(f"            live would need "
          f"{', '.join(example.missing_for_live())}")

tradefloor 0.6.0
mode        replay
            live would need TRADEFLOOR_LIVE_EXAMPLES=1, OPENAI_API_KEY


## 2. The market

Four synthetic instruments. The tickers, the sectors and every fundamental
are generated, so nothing the model knows about real listed companies applies
-- which is the point. An agent cannot recall this market's history, because
it does not have one.

In [2]:
roster = example.universe()
for instrument in roster:
    print(f"{instrument.ticker:10} {instrument.sector:20} "
          f"${instrument.initial_price:7.2f}  "
          f"growth {instrument.revenue_growth:5.0%}  "
          f"ADV {instrument.avg_volume:,.0f}")
print(f"\nseed {example.SEED}, {example.DAYS} days, "
      f"{len(roster)} instruments")

TECH_A     technology           $ 140.00  growth   30%  ADV 5,000,000
TECH_B     technology           $  95.00  growth   22%  ADV 5,000,000
BANK_A     financial_services   $  60.00  growth    5%  ADV 5,000,000
STAPLE_A   consumer_staples     $  48.00  growth    2%  ADV 5,000,000

seed 4242, 5 days, 4 instruments


## 3. The agent

An `agents.Agent` and nothing more. The adapter takes it positionally, the
way every adapter in this package takes its framework object.

The one change the adapter makes is to bind Tradefloor's decision contract as
the agent's `output_type`, and it does that on `Agent.clone(...)` -- so the
original keeps whatever output type it had, which here is none at all.

The standing brief is sent as a *message*, never written into `instructions`.
Overwriting somebody's system prompt would evaluate a different agent from
the one they wrote.

In [3]:
if LIVE:
    from agents import Agent

    pm = Agent(
        name="Portfolio Manager",
        instructions=(
            "You manage a concentrated equity book in a simulated market. "
            "You prefer buying weakness and trimming strength, you size "
            "positions against the limits the observation states, and you "
            "would rather do nothing than force a trade."
        ),
        model=example.LIVE_MODEL,
    )
    agent = OpenAIAgentsAdapter(pm, mode="live", recorder=Transcript(),
                                max_turns=example.LIVE_MAX_TURNS, arm="live")
    agent.recorder.meta.update(agent.provenance())
    print(f"live: {pm.name} on {example.LIVE_MODEL}")
else:
    pm = None
    transcript = Transcript.load(example.FIXTURE)
    agent = OpenAIAgentsAdapter(mode="replay", transcript=transcript,
                                arm="replay")
    print(f"replaying {len(transcript)} recorded interactions from")
    print(f"  {example.FIXTURE.name}\n")
    for field in ("framework", "framework_version", "framework_url",
                  "entry_point", "provider", "model", "agent_name", "mode",
                  "generation", "decision_every_steps",
                  "decision_schema_version", "instructions_version",
                  "instructions_digest", "recorded_utc"):
        print(f"  {field:<25} {transcript.meta.get(field)}")

replaying 5 recorded interactions from
  five-days.json

  framework                 openai-agents
  framework_version         0.22.0
  framework_url             https://github.com/openai/openai-agents-python
  entry_point               agents.Runner.run
  provider                  openai
  model                     gpt-5.2
  agent_name                Portfolio Manager
  mode                      live
  generation                {'max_turns': 6, 'tracing': False}
  decision_every_steps      6
  decision_schema_version   1
  instructions_version      1
  instructions_digest       ca0015e5c92204a3
  recorded_utc              2026-08-31T02:38:06+00:00


Every field a replayed run needs to describe itself is populated: what
framework, what version, what entry point, what model, what generation
settings, and digests of the instructions. Nowhere in that structure is
there a place to put a credential, which is deliberate -- this dictionary is
printed and written into artifacts.

## 4. What the model is actually sent

This is the cell worth reading twice. `payload_of` reads the serialized
observation back out of the exact call the adapter made, so what follows is
not a reconstruction -- it is the input, recovered from the recording.

Two messages go to the model: the standing brief, and the observation as
JSON. Everything the agent knows about this market is below.

In [4]:
first = agent.transcript.entries[0] if not LIVE else None
sample = payload_of(first["prompt"]) if first else None

if sample is None:
    print("live run: the payload is built at the first decision point")
else:
    print("MESSAGE 1 -- the standing brief\n")
    print(BRIEF)
    print("MESSAGE 2 -- the observation, in full\n")
    print(json.dumps(sample, indent=2)[:2600])

MESSAGE 1 -- the standing brief

You are trading a simulated market. Every instrument is synthetic: the tickers, the sectors and the fundamentals are generated, so anything you know about real listed companies does not apply here.

The message after this one is the entire observation, as JSON. You have no other data source and no browsing. Where a field is null it is genuinely unknown -- a five-day return is null until five days have been observed -- and null is not zero.

Seek attractive risk-adjusted returns while controlling downside risk. You may buy, sell, resize or maintain positions, and you are not required to trade: leaving the book alone is a decision.

Two separate limits bound your size and you must respect both. `max_order_shares` is what this market can absorb in one order without your own trade moving the price against you; a larger request is clipped, and the clip is recorded against you rather than silently applied. `portfolio.buying_power` is the additional gross noti

Note what the observation carries and what it does not. Prices, the top of
book, average volume, the agent's own position, the macro state, and **two
different size limits**. No fair value, no factor attribution, no mispricing,
no future macro path. Section 8 checks that rather than trusting it.

## 5. The run

The market advances every step; the agent is asked once a day. On the steps
in between, the adapter records the prices it saw and returns nothing -- a
manager watches the book continuously and revisits it on a schedule.

In [5]:
scores = tf.evaluate({"pm": agent}, seed=example.SEED, universe=roster,
                     days=example.DAYS)
card = scores["pm"]

print(f"decisions   {len(agent.record)}")
print(f"trades      {card.trades}")
print(f"rejected    {card.rejected}")
print(f"turnover    {card.turnover:,.0f}")
print(f"pnl         {card.pnl:+,.0f}")
print(f"return      {card.return_pct:+.2f}%")
print(f"impact      {card.impact_bps:+.2f} bps")

decisions   5
trades      3
rejected    1
turnover    1,676,790
pnl         +20,370
return      +2.04%
impact      +4.24 bps


In [6]:
# The agent's own object is untouched: the contract was bound on a clone.
if LIVE:
    print(f"pm.output_type after the run: {pm.output_type}")
else:
    print("replay: no agent object was involved, which is the point --")
    print("a recorded run needs no key, no network and no SDK install.")
    print(f"'agents' imported: {'agents' in sys.modules}")

replay: no agent object was involved, which is the point --
a recorded run needs no key, no network and no SDK install.
'agents' imported: False


## 6. What it decided, and why

The rationale is the model's own, recorded verbatim. It never affects
execution -- only the actions do -- but it is the thing a backtest cannot
give you: a statement of reasoning you can check against what the market
actually did.

In [7]:
for entry in agent.record:
    acted = ", ".join(f"{a['side']} {a['quantity']:,.0f} {a['symbol']}"
                      for a in entry["decision"]["actions"]
                      if a["side"] != "HOLD") or "no change"
    print(f"day {entry['day']}  {acted}")
    print(f"        \"{entry['decision']['rationale']}\"")
    if entry["clipped"]:
        print(f"        clipped: {entry['clipped']}")
    print()

day 0  no change
        "Day 0 has no realized returns, volatility, or fundamentals, so there’s no edge to “buy weakness/trim strength” yet. With low macro volatility and ample buying power, the best risk-controlled choice is to wait for a few observations before initiating concentrated positions."

day 1  BUY 12,000 TECH_B, BUY 18,000 BANK_A
        "Buy weakness in TECH_B and BANK_A on the day’s drawdowns, sizing modestly within max_order_shares and well inside buying power to control downside while building exposure."

day 2  SELL 3,000 TECH_B
        "Trim TECH_B into strength (up 1d) to reduce leverage and free buying power while keeping a core position given the weaker 5d trend. No new buys: limited buying power and no standout weakness signal beyond a modest dip in TECH_A."

day 3  no change
        "No clear weakness to add into: TECH_B is up on the day and only modestly down 5d, while other names are extended to the upside; with low vol and an expansion/low-VIX backdrop, I pr

## 7. Two size limits, and which one binds

An observation that names only one size limit is a trap, and this is the
concrete, teachable property of the environment.

- **`max_order_shares`** is the *participation* cap: how much of one name the
  MARKET can absorb in a single order without the order itself moving the
  price. It is 2% of average daily volume.
- **`portfolio.buying_power`** is the *funding* cap: how much additional
  gross notional this BOOK can carry before the leverage limit refuses the
  trade.

They are unrelated numbers, and **which one binds is a property of the book,
not a rule**. The participation cap is fixed by trading volume; buying power
scales with equity. So a small book against liquid names is limited by what
it can fund, and a large book against thin names is limited by what the
market can absorb. Both are stated precisely so the agent does not have to
guess which regime it is in.

The payload used to state only the participation cap, and four independent
agents -- two frontier models among them -- sized to it, were refused at a
limit the observation never mentioned, and scored zero trades.

The arithmetic below is computed from the committed observation rather than
quoted, including the equity at which the two limits swap places:

In [8]:
pf = sample["portfolio"] if sample else None
if pf is None:
    print("live run: rerun in replay mode to see the recorded arithmetic")
else:
    participation = sum(a["max_order_shares"] * a["price"]
                        for a in sample["assets"])
    print(f"equity                        {pf['net_worth']:>14,.0f}")
    print(f"leverage cap                  {pf['max_leverage']:>14.1f}x")
    print(f"buying power (funding cap)    {pf['buying_power']:>14,.0f}")
    print(f"max_order_shares, all four    {participation:>14,.0f}"
          f"   <- participation cap")
    print()
    # Which limit binds is derived, never asserted: on a bigger book the
    # inequality flips, and prose that assumed one answer would contradict
    # the numbers printed directly above it.
    crossover = participation / pf["max_leverage"]
    funding_binds = participation > pf["buying_power"]
    print(f"Sizing every name to its participation cap asks for "
          f"{participation / pf['net_worth']:.1f}x equity, against a "
          f"{pf['max_leverage']:.0f}x limit.")
    print(f"On THIS book the binding limit is "
          f"{'buying power' if funding_binds else 'the participation cap'}.")
    print(f"They swap at about {crossover:,.0f} of equity: below that the "
          f"funding cap binds,")
    print("above it the market's capacity does. Neither is the rule; the "
          "book decides.")

equity                             1,000,000
leverage cap                             2.0x
buying power (funding cap)         2,000,000
max_order_shares, all four        34,300,000   <- participation cap

Sizing every name to its participation cap asks for 34.3x equity, against a 2x limit.
On THIS book the binding limit is buying power.
They swap at about 17,150,000 of equity: below that the funding cap binds,
above it the market's capacity does. Neither is the rule; the book decides.


The brief is the other half of the trap, and this notebook has now been
recorded three times against `gpt-5.2` on this same seed and roster while
that brief changed. The results are worth setting out honestly, because
they do not all point the same way.

| brief says | trades | rejected | worst overshoot |
|---|---|---|---|
| size against `max_order_shares` only | 8 | 1 | **3.02x** vs 2.00x -- 51% over |
| both limits, "the funding limit is usually the binding one" | 8 / 10 | 1 / 0 | 2.01x -- 0.5% over |
| both limits, no claim about which binds *(committed)* | 3 | 1 | 2.19x -- 9.3% over |

The robust finding is the first row against the rest: naming only the
participation cap produced an order **half again** the limit, and naming both
brought every subsequent run inside 10%. That reproduces.

The rest does not, yet. Two runs carried the hint that funding usually binds;
one overshot by 0.5% and one not at all. The committed run, with the hint
removed, overshot by 9.3%. That is one sample per configuration and a model
that answers differently every time -- enough to see the big effect, nowhere
near enough to attribute the small one.

**The hint was removed anyway, and for a reason that is not about
performance.** "The funding limit is usually the binding one" is false on a
larger book: the participation cap is fixed by volume, buying power scales
with equity, and above the crossover printed above the inequality flips. A
harness that ships a false generalisation because it happens to nudge the
model helpfully on the roster where it is true is shaping the behaviour it
then measures, which is the one thing this environment exists not to do. The
model is given both numbers and can determine which binds.

In [9]:
print(f"{'day':>3} {'buying power':>14} {'bought':>13} {'sold':>12} "
      f"{'net':>13} {'headroom':>12}")
for entry in agent.record:
    p = payload_of(entry["prompt"])
    price = {a["symbol"]: a["price"] for a in p["assets"]}
    buys = sum(a["quantity"] * price[a["symbol"]]
               for a in entry["decision"]["actions"] if a["side"] == "BUY")
    sells = sum(a["quantity"] * price[a["symbol"]]
                for a in entry["decision"]["actions"] if a["side"] == "SELL")
    power, net = p["portfolio"]["buying_power"], buys - sells
    print(f"{entry['day']:>3} {power:>14,.0f} {buys:>13,.0f} {sells:>12,.0f} "
          f"{net:>13,.0f} {power - net:>12,.0f}"
          f"{'   <-- over' if net > power else ''}")

print()
print(f"rejected orders: {card.rejected}")
for err in card.errors:
    print(f"  {err}")

day   buying power        bought         sold           net     headroom
  0      2,000,000             0            0             0    2,000,000
  1      2,000,000     2,186,940            0     2,186,940     -186,940   <-- over
  2        894,200             0      281,850      -281,850    1,176,050
  3      1,180,550             0            0             0    1,180,550
  4      1,165,430       278,370            0       278,370      887,060

rejected orders: 1
  step 6: trade would take leverage to 2.19x, above the 2.00x limit


Day 1 is the row that missed. On a fresh book with 2,000,000 of buying
power the model ordered 2,186,940 across two names -- 9.3% over -- and the
market refused the decision. Its own rationale for that step says it was
"sizing modestly within `max_order_shares` and well inside buying power",
which it was not; the participation cap it checked was slack, and the
funding cap it believed it had checked was the binding one.

Everything after it is conservative: the model trims on day 2, sits out day
3, and adds a single small position on day 4, finishing with 3 trades where
earlier runs made 8 and 10. A refused order is information the agent
receives, and this one visibly changed how it sized for the rest of the run.

`buying_power` is also a snapshot taken when the observation is built. The
orders execute afterwards against a book whose prices have moved, so even an
agent that sizes exactly to the stated cap can cross it -- which is why the
market checks leverage at execution rather than trusting the plan.

The refusal is the *market's*, not the adapter's, and it names the number.
The decision is recorded in full either way, so the trace shows what the
agent asked for beside what it got -- an experiment that silently dropped
the unaffordable leg would score a plan the agent never made, and a refused
order is not the same thing as a considered decision to hold.

## 8. What it was never shown

`Observation` carries `.engine`, and the engine knows the answer key: fair
value, the nine-way factor attribution of every price move, each company's
mispricing, and the macro path the run has not reached yet. An agent reading
any of it inverts the simulator, and the experiment around it measures
nothing.

So the observation mapping is an **allowlist, written out field by field**.
Below, the fair value of every instrument is computed from public
fundamentals and the macro state the model *was* shown -- and then every
recorded prompt is scanned for it.

In [10]:
macro = sample["macro"]
hidden = {}
for instrument in roster:
    hidden[instrument.ticker] = tf.fair_value(
        eps=instrument.eps,
        sector=instrument.sector,
        revenue_growth=instrument.revenue_growth,
        book_value_per_share=instrument.book_value_per_share,
        federal_funds_rate=macro["federal_funds_rate"],
        corporate_bond_yield=macro["corporate_bond_yield"],
    ).fair_value

sent = json.dumps([e["prompt"] for e in agent.transcript.entries])

print(f"{'ticker':10} {'price shown':>12} {'fair value':>12}   in the prompt?")
for a in sample["assets"]:
    value = hidden[a["symbol"]]
    leaked = f"{value:.4f}" in sent or f"{value:.2f}" in sent
    print(f"{a['symbol']:10} {a['price']:>12,.2f} {value:>12,.2f}   "
          f"{'LEAKED' if leaked else 'absent'}")

names = [w for w in ("fair_value", "mispricing", "attribution", "crowd_lean")
         if w in sent.lower()]
print(f"\nforbidden field names in the prompt: {names or 'none'}")
print(f"prompt bytes scanned: {len(sent):,}")

ticker      price shown   fair value   in the prompt?
TECH_A           140.00        94.71   absent
TECH_B            95.00        94.84   absent
BANK_A            60.00        35.67   absent
STAPLE_A          48.00        59.48   absent

forbidden field names in the prompt: none
prompt bytes scanned: 17,114


The model priced this book without ever being told what the simulator thinks
the book is worth. That is what makes the comparison mean something: the
agent inferred, it was not informed.

`tests/test_openai_agents.py` proves the same boundary twice on every run --
once against an engine proxy that raises if the forbidden surface is
*touched*, and once by scanning what the model actually received, which is
the check repeated above.

## 9. Reproducing this

The recording is committed at `tests/fixtures/openai_agents/five-days.json`.
Replaying it needs no API key, no network, and not even the SDK installed --
the market re-executes for real and only the agent's answers come from the
file, keyed by a digest of the exact input.

Change the observation mapping, the brief or the market and the digest moves,
the key goes missing, and the replay **refuses** naming the step rather than
answering the new question with an answer given to the old one. That is not a
hypothetical: rewriting the brief to name both size limits invalidated the
first recording, which is how the fixture above came to be re-recorded.

In [11]:
same = tf.evaluate(
    {"pm": OpenAIAgentsAdapter(mode="replay",
                               transcript=Transcript.load(example.FIXTURE))},
    seed=example.SEED, universe=example.universe(), days=example.DAYS)["pm"]

print(f"this notebook   {card.pnl:+,.0f}")
print(f"fresh replay    {same.pnl:+,.0f}")
print(f"identical       {same.pnl == card.pnl}")

# Asserted, not merely printed. `tf.evaluate` scores many agents and so
# ABSORBS a refusal into `card.errors` rather than raising -- correct for a
# harness, exactly wrong for a notebook demonstrating one run. A corrupted
# recording would make some decisions fail to replay, and this notebook
# would print a smaller, plausible, wrong result and every cell would still
# be green. These three lines are what make that impossible.
assert same.pnl == card.pnl, "the replay did not reproduce this run"
assert len(agent.record) == example.DAYS, (
    f"only {len(agent.record)} of {example.DAYS} decisions replayed")
assert card.rejected <= 1 and not [
    e for e in card.errors if "leverage" not in e], card.errors

if not LIVE:
    print()
    print(f"recorded with   {agent.transcript.meta.get('model')} "
          f"({agent.transcript.meta.get('provider')}) on "
          f"{agent.transcript.meta.get('recorded_utc')}")

this notebook   +20,370


fresh replay    +20,370
identical       True

recorded with   gpt-5.2 (openai) on 2026-08-31T02:38:06+00:00


---

**Where to go next.** `examples/integrations/openai_agents/five_days.py` is the
same experiment as a script, using the SDK's own deterministic model so it
runs offline in seconds. `examples/integrations/finrobot/rate_shock.ipynb`
takes a different shape entirely: a checkpoint, a fork, and one macro
intervention applied to a single arm, which is how you ask what an agent
would have done otherwise.